# Starter (Follow‑Along): Web Scraping & API Integration

**Pulling data from e‑commerce sites with BeautifulSoup + Selenium + XPath + API Integration**

This is the **fill‑in‑the‑blanks** version of `walkthrough.ipynb`. The structure and the
explanations are identical, but **every code cell contains only comments / steps** — no code yet.

**How to use it (in class):**
- Read the markdown for each section, then write the code together, one step (`# TODO`) at a time.
- The full solution lives in `walkthrough.ipynb` if you get stuck or want to copy‑paste.

**Who it's for:** participants who are new to web scraping & APIs.

**Prerequisites:**
- Understand Python basics (variables, lists, dicts, loops, functions).
- Already ran `uv sync` in the project root (see `README.md`).
- Chrome/Chromium browser installed (for the Selenium section).

**By the end, you will be able to:**
- Understand the flow of a simple data pipeline.
- Scrape static websites with **BeautifulSoup**.
- Scrape dynamic websites with **Selenium** + **XPath**.
- Fetch data from an **API**.
- Clean data with **pandas** and store it in a **database**.

## Outline

0. Setup & imports
1. Understanding the Data Pipeline
2. BeautifulSoup — scrape 1 item
3. BeautifulSoup — 1 page → list of dictionaries
4. BeautifulSoup — multiple pages (pagination)
5. Selenium — dynamic websites
6. XPath
7. API
8. Data cleaning (pandas)
9. Save to database (SQLite)
10. Exercises + pitfalls

### The pipeline we are building

```
Scrape 1 item → Scrape 1 page → List of Dictionaries → Multiple Pages
   → Data Cleaning → Database Connection → Create Table & Insert
```

### Data sources

| Technique              | Website                          |
| ---------------------- | -------------------------------- |
| BeautifulSoup (static) | https://books.toscrape.com       |
| Selenium (dynamic)     | https://quotes.toscrape.com/js   |
| API                    | https://dummyjson.com/products   |


In [ ]:
# 0. Setup — import everything we'll use in this notebook.
# TODO: standard library
#   - re       (regex, later for cleaning "£51.77")
#   - sqlite3  (built-in database)
# TODO: third-party
#   - pandas as pd            (DataFrame / data cleaning)
#   - requests               (HTTP GET)
#   - BeautifulSoup from bs4  (HTML parsing)
# TODO: print a short "ready" message to confirm the imports worked.


## 1. Understanding the Data Pipeline

A **data pipeline** is an **automated** flow for moving data from a **source** to a
**destination**, through a process that is **structured** and **repeatable**.

In this notebook we build a pipeline: fetch data from the web → clean it → store it in a database.

---

## 2. BeautifulSoup — Scrape 1 Item

`BeautifulSoup` is a library for *parsing* HTML, so we can extract data
based on tags/attributes. The two main methods are:

- `find()` → grabs **one** element
- `find_all()` → grabs **multiple** elements

We start with a single item (one book) on `books.toscrape.com`.


In [ ]:
# 2. BeautifulSoup — Step 1: make an HTTP GET request to the target site.
# TODO: url = the books.toscrape.com homepage
# TODO: response = requests.get(url, timeout=10)
# TODO: response.raise_for_status()      # stop early if the site returned an error (4xx/5xx)
# TODO: response.encoding = "utf-8"       # so the £ symbol displays correctly


In [ ]:
# 2. BeautifulSoup — Step 2: parse the HTML and extract ONE item (one book).
# TODO: (re)fetch the page: url / requests.get(timeout=10) / raise_for_status() / encoding="utf-8"
# TODO: soup = BeautifulSoup(response.text, "html.parser")
#
# find() -> grabs the FIRST matching element only.
# TODO: item = the first  <article class="product_pod">
#
# From that single item, pull out each field:
# TODO: title  -> inside <h3> ... <a>,  read the ["title"] attribute
# TODO: price  -> <p class="price_color">        .text
# TODO: rating -> <p class="star-rating">        ["class"][1]   (e.g. "Three")
# TODO: stock  -> <p class="instock availability">  .text.strip()
#
# TODO: print each field to check the result.


In [ ]:
# (Optional) Practice chaining find() / find_all().
# TODO: from the sidebar <ul class="nav-list">, drill down to the category links:
#         .find("li").find("ul").find_all("a")
# TODO: loop over them and print each category name  (.text.strip())


## 3. Scrape 1 Page → List of Dictionaries

Now we grab **all** products on a single page with `find_all()`, then store
each product as a **dictionary**, collected into a **list**.


In [ ]:
# 3. Scrape ONE page -> a LIST of DICTIONARIES.
# TODO: def scrape_page(url: str) -> list[dict]:
#         - requests.get(url, timeout=10) / raise_for_status() / encoding = "utf-8"
#         - soup = BeautifulSoup(resp.text, "html.parser")
#         - products = []
#         - for item in soup.find_all("article", class_="product_pod"):   # ALL books now
#               products.append({
#                   "name":   ...  # h3 > a  ["title"]
#                   "price":  ...  # p.price_color  .text
#                   "rating": ...  # p.star-rating  ["class"][1]
#               })
#         - return products
#
# TODO: one_page_data = scrape_page("https://books.toscrape.com/")
# TODO: print how many products you got, then peek at  one_page_data[:3]


## 4. Scrape Multiple Pages (Pagination)

Page N follows the URL pattern `https://books.toscrape.com/catalogue/page-{N}.html`.
We loop over the page numbers and combine the results into one big list.

In [ ]:
# 4. Scrape MULTIPLE pages (pagination).
# Page N follows this URL pattern:  https://books.toscrape.com/catalogue/page-{}.html
#
# TODO: reuse scrape_page() from the previous step (paste it above/here if needed).
# TODO: BASE_URL = the "page-{}" pattern above
# TODO: def scrape_multiple_pages(num_pages: int = 3) -> list[dict]:
#         - all_products = []
#         - for page in range(1, num_pages + 1):
#               url = BASE_URL.format(page)
#               all_products.extend(scrape_page(url))
#         - return all_products
#
# TODO: raw_data = scrape_multiple_pages(num_pages=3)
# TODO: print the total count, then peek  raw_data[:3]


In [ ]:
# Why do we need Selenium? First, try the JS-rendered site with plain requests.
# TODO: requests.get("https://quotes.toscrape.com/js/") / raise_for_status() / encoding="utf-8"
# TODO: soup = BeautifulSoup(resp.text, "html.parser")
# TODO: print(soup)  ->  notice the quotes are NOT in the HTML (JavaScript builds them later).


## 5. Selenium — Dynamic Websites

Some websites build their content with **JavaScript**. For example
`https://quotes.toscrape.com/js/` — if you fetch it with `requests`, the content is empty.

`Selenium` controls a **real browser**, so JavaScript runs as well.

> The cell below will **open a Chrome window** briefly (default `headless=False` so it is
> visible during the demo). The driver is downloaded automatically by **Selenium Manager**, so
> there is no need to install ChromeDriver manually. Set `headless=True` if you don't want a window to open.

In [ ]:
# 5. Selenium — drive a REAL browser so the JavaScript runs.
# TODO: imports
#   from selenium import webdriver
#   from selenium.webdriver.chrome.options import Options
#   from selenium.webdriver.common.by import By
#   from selenium.webdriver.support.ui import WebDriverWait
#   from selenium.webdriver.support import expected_conditions as EC
#   import time
#
# TODO: def build_driver(headless: bool = False) -> webdriver.Chrome:
#         - options = Options()
#         - if headless: options.add_argument("--headless=new")
#         - options.add_argument("--window-size=1920,1080")
#         - return webdriver.Chrome(options=options)   # Selenium Manager downloads the driver for you
#
# TODO: driver = build_driver()
# TODO: try:
#         - driver.get("https://quotes.toscrape.com/js/")
#         - # BAD: time.sleep(3)  -> arbitrary fixed delay (too slow, or not enough)
#         - # GOOD: wait only until a .quote element actually appears (up to 10s):
#           WebDriverWait(driver, 10).until(
#               EC.presence_of_element_located((By.CLASS_NAME, "quote")))
#         - quotes = driver.find_elements(By.CLASS_NAME, "quote")
#         - for each quote: read .text (By.CLASS_NAME "text") and author (By.CLASS_NAME "author")
#       finally:
#         - driver.quit()   # ALWAYS close the browser


## 6. XPath

**XPath** is a query language for selecting elements inside the DOM (HTML/XML).
You can try it at [xpath-playground](https://scrapinghub.github.io/xpath-playground/).

A few examples:

| XPath                          | Meaning                                    |
| ------------------------------ | ------------------------------------------ |
| `//h1`                         | all `<h1>` tags                            |
| `//p[1]`                       | the first `<p>` tag                        |
| `//*[@id="first-name"]`        | the element with `id="first-name"`         |
| `//p[@class="plot"]`           | `<p>` with class exactly `plot`            |
| `//p[contains(@class,"plot")]` | `<p>` whose class **contains** `plot`      |

In Selenium, we use `By.XPATH` to find elements.

In [ ]:
# 6. XPath — same task, but locate elements with XPath instead of class names.
# TODO: driver = build_driver()
# TODO: try:
#         - driver.get("https://quotes.toscrape.com/js/")
#         - quotes = driver.find_elements(By.XPATH, '//div[@class="quote"]')      # absolute
#         - for each quote (RELATIVE xpath starts with "." -> search INSIDE the quote):
#               text   = q.find_element(By.XPATH, './/span[@class="text"]').text
#               author = q.find_element(By.XPATH, './/small[@class="author"]').text
#         - also try an absolute one:  driver.find_element(By.XPATH, "//h1").text
#       finally:
#         - driver.quit()


## 7. API

An **API** is the official way to fetch structured data directly from the provider's system.
The data is usually in **JSON** form, stable, and faster than scraping.

We use a public e-commerce API: `https://dummyjson.com/products`.

In [ ]:
# 7. API — fetch structured data directly (JSON, no scraping needed).
# TODO: resp = requests.get(
#           "https://dummyjson.com/products",
#           params={"limit": 5, "select": "title,price,category,brand"},
#           timeout=10,
#       )
# TODO: resp.raise_for_status()
# TODO: api_data = resp.json()             # JSON -> Python dict
# TODO: api_products = api_data["products"]
# TODO: print how many products, then inspect  api_products


## 8. Data Cleaning (pandas)

The scraped data is still "dirty": price is text `"£51.77"`, rating is a word `"Three"`.
We clean it with **pandas** before saving:

1. Load it into a `DataFrame`
2. Convert price `"£51.77"` → `51.77` (float)
3. Convert rating `"Three"` → `3` (int)
4. Drop empty rows & duplicates

In [ ]:
# 8. Data Cleaning with pandas.
# The scraped data is still "dirty": price is text "£51.77", rating is a word "Three".
# TODO: RATING_MAP = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
# TODO: df = pd.DataFrame(raw_data)
# TODO: price  "£51.77" -> 51.77 :
#         df["price"] = df["price"].apply(lambda x: re.sub(r"[^0-9.]", "", x)).astype(float)
# TODO: rating "Three"  -> 3     :
#         df["rating"] = df["rating"].map(RATING_MAP)
# TODO: drop empty rows & duplicates :
#         df = df.dropna().drop_duplicates().reset_index(drop=True)
# TODO: check df.dtypes and df.head()


## 9. Save to Database (SQLite)

The final step: save the clean data to a database. We use **SQLite** (built into Python,
no server to install). The flow is: **create connection → create table → insert data**.

> A **PostgreSQL** version (`psycopg2`) is in `steps/09_simpan_database.py` if you want
> to demonstrate a database server.

In [ ]:
# 9. Save to a database (SQLite): create connection -> create table -> insert.
# TODO: DB_PATH = "walkthrough.db"
# TODO: conn = sqlite3.connect(DB_PATH)
# TODO: conn.execute("""
#           CREATE TABLE IF NOT EXISTS products (
#               id     INTEGER PRIMARY KEY AUTOINCREMENT,
#               name   TEXT NOT NULL,
#               price  REAL,
#               rating INTEGER
#           )
#       """)
# TODO: conn.execute("DELETE FROM products")   # clear first so re-running doesn't duplicate rows
# TODO: df.to_sql("products", conn, if_exists="append", index=False)
# TODO: conn.commit()
# TODO: read it back:  pd.read_sql("SELECT * FROM products LIMIT 5", conn)
# TODO: conn.close()


## 10. Exercises

1. Modify `scrape_multiple_pages` to fetch **5 pages**, then compute the **average price**
   of the books using pandas.
2. (Bonus) Add a **link** column to the scraped results, then save it to the database table.

Try it yourself before looking at the example answer in the next cell.

In [ ]:
# Exercise 1 — your turn (try before peeking at walkthrough.ipynb):
# TODO: scrape 5 pages ->  scrape_multiple_pages(num_pages=5)
# TODO: load into a DataFrame and clean the price column (same re.sub trick as Step 8)
# TODO: print the number of books and the AVERAGE price  (df["price"].mean())
#
# Bonus: add a "link" column to the scraped dict, then save it to the DB table too.


In [ ]:
# (Optional extension) a sturdier price parser than the regex above:
# !pip install price_parser
# from price_parser import Price
# Price.fromstring("£51.77").amount_float   # -> 51.77


## Pitfalls & Extensions

**Common mistakes:**
- Forgetting to check `response.raise_for_status()` → silently processing an error page.
- `find()` returns `None` when the element is missing → it will error on `.text`. Always make sure the structure is correct.
- Forgetting `driver.quit()` → many browser processes stuck in memory.
- Scraping too fast / aggressively → respect the site (add delays, read `robots.txt`).

**Extensions:**
- Run the full pipeline from the terminal: `uv run python pipeline.py --pages 5`.
- Add `time.sleep()` between requests, or use `WebDriverWait` to wait for elements to appear.
- Save to **PostgreSQL** (see `steps/09_simpan_database.py`).